# 05 Measure Relationships

Assignment step 8 bonus: generate grounded, typed candidate relationships between final measures. `narrower_than` is represented as the inverse reading of exported `broader_than` edges.


In [1]:
from pathlib import Path
import csv
import json

import pyarrow.parquet as pq

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent


def read_json(relative_path: str):
    path = ROOT / relative_path
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"missing": str(path)}


def csv_rows(relative_path: str, limit: int | None = None):
    csv.field_size_limit(2_147_483_647)
    path = ROOT / relative_path
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        rows = list(csv.DictReader(file))
    return rows if limit is None else rows[:limit]


def csv_count(relative_path: str) -> int | None:
    path = ROOT / relative_path
    if not path.exists():
        return None
    return len(csv_rows(relative_path))


def parquet_count(relative_path: str) -> int | None:
    path = ROOT / relative_path
    return pq.read_table(path).num_rows if path.exists() else None


## Relationship Summary


In [2]:
metrics = read_json("report/relations_metrics.json")
{
    "relation_count": metrics.get("relation_count"),
    "relation_type_distribution": metrics.get("relation_type_distribution"),
    "validation": metrics.get("validation"),
    "manual_review": metrics.get("manual_review"),
    "latest_run_summary": metrics.get("latest_run_summary"),
}


{'relation_count': 5628,
 'relation_type_distribution': {'broader_than': 442,
  'related_to': 5184,
  'variant_of': 2},
 'validation': {'failures': [], 'passed': True},
 'manual_review': {'completed_count': 0,
  'review_sample': 'outputs\\relations\\run_1b5f0cbf1084aaa8d017\\manual_relation_review_sample.csv',
  'sample_count': 100,
  'status': 'pending'},
 'latest_run_summary': {'accepted_count': 5628,
  'candidate_count': 5628,
  'confidence_bands': {'0.00-0.69': 3909, '0.70-0.84': 1510, '0.85-1.00': 209},
  'generation_method_distribution': {'domains': 5184,
   'embedding_similarity': 5184,
   'lexical_containment': 442,
   'lexical_variant': 2,
   'qualifier_patterns': 442,
   'title_family': 2},
  'llm_adjudication_enabled': False,
  'rejection_distribution': {},
  'relation_type_distribution': {'broader_than': 442,
   'related_to': 5184,
   'variant_of': 2},
  'run_id': 'run_1b5f0cbf1084aaa8d017'}}

## Relationship Examples


In [3]:
csv_rows("outputs/measure_relations.csv", 8)


[{'relation_id': 'relation_04627d6eb77a6477146f',
  'source_term_id': 'term_d3e9231fa5288ba36887',
  'source_term': 'Long-term unemployment rate by sex',
  'target_term_id': 'term_6273fd9149de8fb6704b',
  'target_term': 'Long-term unemployment rates by sex',
  'relation_type': 'related_to',
  'confidence': '0.961893',
  'evidence_ids_json': '["occ_0fbb8efc6aa16476ff76", "occ_2a0fd9c58eb983f3d0e9", "occ_c82d920ac0882b08217e"]',
  'evidence': '{"domain_signal": "same_domain", "embedding_similarity": 0.996577, "source_domain": "labour market", "target_domain": "labour market", "token_jaccard": 0.666667}',
  'generation_methods_json': '["embedding_similarity", "domains"]',
  'run_id': 'run_1b5f0cbf1084aaa8d017'},
 {'relation_id': 'relation_a4f7fca112a2a6cacc79',
  'source_term_id': 'term_aa9dcb3b1b5e3cb0f0d6',
  'source_term': 'Gross value added - NACE Rev. 2: F - current prices',
  'target_term_id': 'term_7f231a9b7cbedf86f343',
  'target_term': 'Gross value added - NACE Rev. 2: L - curren